[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/2-tools-and-loop.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 2. Tools and the agent loop

Tutorial 1 returned one policy section. This notebook keeps that tool, adds the catalog, and wraps the exchange in a loop.

`get_store_fact` still maps a topic to a section. The model does not pass a filename. `query_catalog` reads the SQLite catalog from `tutorial.common.get_db`. The model passes a name or a sku, not a SQL string. Each row has sku, name, stock, price, shippable, and category.

`run_agent` is perceive, reason, act, observe. It stops when the model answers, when the step cap hits, or when the same call repeats.

A customer asks whether Calabrian chili oil is in stock and whether a jar can ship to Ohio. Shared helpers are imported, so this file runs on its own.

## Setup

Find the repo so `import tutorial` works, then load the key. `chat` calls the OpenAI Chat Completions API. The key is not printed.

In [ ]:
import sys
from pathlib import Path


def find_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "tutorial" / "common" / "client.py").is_file():
            return folder
    raise RuntimeError("Open this notebook inside the book-agents repository.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.runtime import require_key

_, model = require_key()
print("model:", model)

## Catalog row

The seed is in `tutorial.common.get_db`. The SQL stays in `tutorial.common.read_db`.

In [ ]:
import tutorial.common.facts  # registers get_store_fact
import tutorial.common.catalog  # registers query_catalog

from tutorial.common.get_db import reset_db
from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent
from tutorial.common.read_db import row_for

reset_db()
chili = row_for("HJ-CHI")
print(chili["name"], "stock", chili["stock"], "shippable", chili["shippable"], "price", chili["price"])

## Run

One customer question, then a step cap, then a repeated lookup. The harness writes the stop. It does not invent the rest of the answer.

In [ ]:
reset_db()
question = "Do you have Calabrian chili oil, and can you ship a jar to Ohio?"

print("=== perceive, reason, act, observe ===")
result = run_agent(question, system_text())
print("ANSWER:", result["text"])
log = "\n".join(result["tool_log"])
check(result["stopped"] == "final", "the loop stopped with a final answer")
check(result["steps"] >= 2, "the loop used more than one model call")
check("query_catalog" in log, "the catalog tool ran")
check("get_store_fact" in log and "shipping" in log.lower(), "the shipping topic was looked up")
check("14" in log, "stock came from SQLite")
check("ohio" in result["text"].lower(), "the answer talks about Ohio")

print("=== max steps ===")
capped = run_agent(question, system_text(), max_steps=1)
print("ANSWER:", capped["text"])
check(capped["stopped"] == "max_steps", "max_steps stops without inventing an answer")

print("=== repeated call ===")
repeated = run_agent(
    "Look up the shipping topic with get_store_fact, then look up that same topic again, then a third time with the same argument. Only then answer me.",
    system_text(),
)
print("ANSWER:", repeated["text"])
check(repeated["stopped"] == "repeated_call", "the third identical call stops the loop")